# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library for structured, reproducible science workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Dataset Identifier: {metadata.identifier}")
print(f"Dataset Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets are listed in the `recordSet` property. Each record set may contain multiple fields, each uniquely identified by its `@id`. All references throughout this notebook will use `@id` values.

In [ ]:
# Overview all record sets and their fields, referenced by their `@id`.

record_set_ids = []
field_info = {}

for record_set in metadata.recordSet:
    rs_id = record_set['@id']
    record_set_ids.append(rs_id)
    print(f"Record Set: {rs_id}")
    field_info[rs_id] = []
    for field in record_set.get('field', []):
        fid = field['@id']
        label = field.get('label', fid)
        dtype = field.get('dataType', 'unknown')
        field_info[rs_id].append((fid, label, dtype))
        print(f"  Field: {fid}, Label: {label}, DataType: {dtype}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Each record set and field is referenced by its `@id`, as shown in the overview above.

In [ ]:
# Extract data from all record sets by their `@id`.

# Use record_set_ids found above
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for Record Set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering, normalization, and grouping.

For demonstration, we'll pick the first record set and identify a numeric field for further analysis using its `@id`.

In [ ]:
# Choose the first record set and a numeric field (example: age or hormone levels).

example_record_set = record_set_ids[0]
df = dataframes[example_record_set]

# Find a numeric field in the selected record set
numeric_field_id = None
for fid, label, dtype in field_info[example_record_set]:
    if 'Integer' in dtype or 'Float' in dtype or 'Number' in dtype or 'age' in label.lower():
        numeric_field_id = fid
        break

print(f"Using numeric field: {numeric_field_id}")

# EDA: filtering records
threshold = 20
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field (e.g., presence of a clinical feature)
    group_field = None
    for fid, label, dtype in field_info[example_record_set]:
        if 'Boolean' in dtype or 'infertility' in label.lower():
            group_field = fid
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the record sets using their `@id`.

We'll plot the distribution of the chosen numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True)
        plt.title(f"Normalized Distribution of {numeric_field_id}")
        plt.xlabel(norm_col)
        plt.ylabel("Frequency")
        plt.show()

## 6. Conclusion
We explored the FAIR^2 Clinical Dataset for genotype–phenotype heterogeneity analysis using `mlcroissant`.

Key findings:
- The dataset contains multiple record sets, each with rich clinical, hormonal, and genetic features referenced by `@id`.
- Data was loaded and processed using Croissant schema structure, ensuring reproducible workflow.
- Filtering, normalization, and grouping allow flexible exploratory analysis even on limited-size datasets.
- Visualizations help reveal distributions and relationships in the data.

Further scientific or clinical research may benefit from expanding the cohort and linking additional features.